# Explore API tour — DPR model

Walkthrough of the new query layer (`aq.explore()`) on the DPR trailer model.

Run the server from `dpr_scripts/` (`acquirium server --config acquirium.toml`).
Everything up to the last section works on the bare model; the timeseries cells
at the end need the CSV driver to have ingested `./raw`.

In [ ]:
from acquirium import Acquirium
from acquirium.Client.explore import (
    Not, Shortcut, Step, SHORTCUTS, register_shortcut, hide, unhide,
)
from acquirium.internals.internals_namespaces import S223

acq = Acquirium(server_url="localhost", server_port=8000)


## 1. Entities

`entity()` takes a class as a URI, a namespace constant, or free text — text is
resolved through the server. Every verb returns a new immutable query, so
variants never interfere.

In [ ]:
tanks = acq.explore().entity("tank", alias="tank")
tanks.metadata()

In [ ]:
## branch from a shared base without side effects
base = acq.explore().entity("Equipment", alias="eq")
pumps = acq.explore().entity("pump", alias="pump")
len(base.metadata()), len(pumps.metadata())

In [ ]:
## pin an exact instance by URI or CURIE
acq.explore().entity(uri="dpr:chlorine-contactor", alias="cl2").metadata()

## 2. Attribute filters — `where()` and keyword sugar

One unified filter for entity and measurement nodes. Values can be free text,
URIs, lists (OR), or `Not(...)` (exclude). `process` filtering works now (the
old builder stored it but never compiled it).

In [ ]:
## equipment running an ozonation process (keyword sugar on entity())
acq.explore().entity("Equipment", alias="eq", process="ozonation").metadata()

In [ ]:
## the same via where(); a list means OR
(acq.explore().entity("Equipment", alias="eq")
 .where(process=["uv disinfection", "biologically active filtration"])
 .metadata())

In [ ]:
## cp_type filters by connection point class
(acq.explore().entity("Equipment", alias="eq", cp_type="bidirectional connection point")
 .metadata())

## 3. Measurements

`measurement()` attaches the data-bearing node (a point with a registered
stream). Attribute keywords filter it in place; without the CSV driver these
return empty because nothing carries an external reference yet.

In [ ]:
## turbidity measurements anywhere on the plant
(acq.explore().entity("Equipment", alias="eq")
 .measurement(alias="turbidity", quantity_kind="turbidity")
 .metadata())

In [ ]:
## water-side flow measurements, excluding air lines
flows = (acq.explore().entity("Equipment", alias="eq")
         .measurement(alias="flow", quantity_kind="volume flow rate",
                      medium=Not("air")))
flows.metadata()

## 4. Projected columns — `include()`

Additive: each attribute becomes an `alias.attr` metadata column, bound
OPTIONALly so rows without the attribute survive. `of=` targets any alias.

In [ ]:
flows.include("medium", "unit").metadata()

In [ ]:
## project the equipment's process next to its measurements
flows.include("medium").include("process", of="eq").metadata()

## 5. `refocus()`

Moves the builder pointer back to an earlier alias, e.g. to hang two
measurements off the same entity.

In [ ]:
(acq.explore().entity("biological aerated filter", alias="baf")
 .measurement(alias="nh3", substance="ammonia")
 .refocus("baf")
 .measurement(alias="flow", quantity_kind="volume flow rate")
 .metadata())

## 6. Traversal — via expressions and shortcuts

`related(via=...)` accepts `"any"`, a bare predicate (URI or free text), a
shortcut name, or a composition: segments joined with `/`, repeated with `*`.
One shortcut step = one *meaningful* hop, however deep the RDF plumbing is.

In [ ]:
for s in SHORTCUTS.values():
    print(f"{s.name:24s} {s.description}")

In [ ]:
## bare predicate via free text: which sensor observes which property
(acq.explore().entity("Sensor", alias="sensor")
 .related("QuantifiableObservableProperty", alias="observed", via="observes")
 .metadata())

In [ ]:
## one meaningful step downstream of the ozone unit
(acq.explore().entity("ozonation unit", alias="o3")
 .related("Equipment", alias="next", via="downstream_equipment")
 .metadata())

In [ ]:
## composition with repetition: up to 3 equipment steps downstream,
## then that equipment's properties (0 repetitions = the unit's own properties)
(acq.explore().entity("ozonation unit", alias="o3")
 .related("QuantifiableObservableProperty", alias="prop",
          via="downstream_equipment*/downstream_property", max_depth=4)
 .metadata())

In [ ]:
## your own shortcut; Step predicates and node classes can be free text too
register_shortcut(Shortcut(
    "member",
    ((Step(str(S223.hasMember)),),),
    "System to its member entities.",
))
(acq.explore().entity("System", alias="sys")
 .related("Equipment", alias="member", via="member")
 .include("type", of="member")
 .metadata())

## 7. Hidden predicates

Attribute predicates (`rdf:type`/`subClassOf`, media, process, `hasProperty`,
external refs, ...) and `s223:cnx` are hidden from `via="any"` traversal by
default — they are node attributes, not plant edges, and traversing them walks
into hub nodes or the ontology TBox. Measurement edges are exempt (that's how
data attaches). Naming a predicate in `via=` always overrides hiding;
`unhide()` lifts defaults, bare `unhide()` resets.


In [ ]:
from acquirium.Client.explore import hidden_predicates
sorted(hidden_predicates())

In [ ]:
## cnx is hidden by default; unhide it to see the noise it would add
clean = (acq.explore().entity("pump", alias="pump")
         .related("Junction", alias="next", max_depth=1).metadata())
unhide(S223.cnx)
noisy = (acq.explore().entity("pump", alias="pump")
         .related("Junction", alias="next", max_depth=1).metadata())
unhide()  ## back to defaults
len(clean), len(noisy)

## 8. Nearest matches — client-side BFS

`nearest=True` resolves the edge by BFS over the via program: per source, only
the closest match survives (equal-distance ties are all kept). Target
constraints participate in nearness — a closer non-matching node does not
shadow a farther matching one. The matches are injected back into the final
SPARQL as paired VALUES, so each source joins only its own nearest target.

In [ ]:
## each UV unit's nearest downstream tank
(acq.explore().entity("ultraviolet light unit", alias="uv")
 .related("tank", alias="tank", via="downstream_equipment*",
          nearest=True, max_depth=6)
 .metadata())

In [ ]:
## nearest upstream pH measurement of the chlorine contactor
(acq.explore().entity(uri="dpr:chlorine-contactor", alias="cl2")
 .measurement(direction="upstream", nearest=True, max_depth=5,
              alias="ph_upstream", quantity_kind="acidity")
 .metadata())

In [ ]:
## to_sparql() previews the pattern; BFS + VALUES happen at execute()
q = (acq.explore().entity("ultraviolet light unit", alias="uv")
     .related("tank", alias="tank", via="downstream_equipment*",
              nearest=True, max_depth=6))
print(q.to_sparql()[:800])

## 9. Faceted exploration — `options()` and `facets()`

`options(attr)` aggregates one attribute's values over the *current* matches
(counts = distinct matched nodes). `facets()` does it for every applicable
attribute at once, falling back to model-wide usage, then to the ontology
vocabulary, when the pattern has no values — the scope tag says which you're
looking at.

In [ ]:
all_meas = acq.explore().entity("Equipment", alias="eq").measurement(alias="m")
all_meas.options("quantity_kind")

In [ ]:
all_meas.options("substance")

In [ ]:
all_meas.facets()

In [ ]:
## facets of an entity node
acq.explore().entity("uv unit", alias="uv").facets()

## 10. Timeseries — needs the CSV driver

`data()` returns the lazy DataObject (unchanged contract): index it by alias,
group it by an entity alias, or flatten with `dataframe()`.

In [ ]:
d = flows.data(cast_value="float")
d

In [ ]:
flows.dataframe(shape="wide", cast_value="float").head()

In [ ]:
## per-equipment grouping
for uri, group in d.by("eq"):
    print(uri)

In [ ]:
## leave the session clean
unhide()